## Paths and executables

In [1]:
conda activate gapseq
gapseq -v
conda deactivate

gapseq version: 1.4.0
Sequence DB md5sum: 954ab58 (2024-11-06, Bacteria)

    The Archaea Sequence Database has not yet been downloaded/updated (default version).
    Get the latest version by running: 'bash ./src/update_sequences.sh Archaea' in your gapseq installation directory.
                        


In [2]:
# Path to new strains' genomes and the panmodel
label="clostridiales"
genomeFolder="00-fastas/new"
pangenome="01-pangenome/$label.faa"
panmodelReactionTable="02-pan_model/$label-all-Reactions.tbl"

# Path to results folder
workFolder="03-strain_models"
modelsFolder=$workFolder/models
matchesFolder=$workFolder/matches
mkdir -p $workFolder $matchesFolder $modelsFolder

# root folder of this analysis
root=$(pwd)

## Parameters

In [3]:
n_cores=14
diamond_bits=150
gapseq_bits=150
biomass="pos"
medium="media/P2_medium_an.csv"

## Derive strain GSMM

### Anticipatory character cleanup

In [ ]:
find $genomeFolder/*.faa | xargs -I % bash -c 'cat % | tr -d "’" > tmp && mv tmp %'

### Identify homologous pangenome sequences

In [ ]:
diamond makedb --in $pangenome -d $label

In [ ]:
dir -1 $genomeFolder | xargs basename -s .faa | xargs -I % \
diamond blastp -d $label -q $genomeFolder/%.faa -o $matchesFolder/%.tsv --min-score $diamond_bits

### Filter panmodel BLAST table

In [4]:
for g in $(dir -1 $genomeFolder | xargs basename -s .faa); do
    mkdir -p $modelsFolder/$g
    echo "Filtering reaction table of $g"
    python utils/filter_blast_table.py $panmodelReactionTable $matchesFolder/$g.tsv $modelsFolder/$g/filtered-Reactions.tbl

    # Readd the header from the panmodel reaction table
    mv $modelsFolder/$g/filtered-Reactions.tbl tmp
    cat <(grep '#' $panmodelReactionTable) tmp > $modelsFolder/$g/filtered-Reactions.tbl
    rm tmp
done

Filtering reaction table of C_acetobutylicum
Filtering reaction table of C_beijerinckii_13821
Filtering reaction table of C_beijerinckii_791


### Construct draft strain model

In [5]:
conda activate gapseq

In [6]:
gapseq_exc=$(which gapseq)

In [7]:
for g in $(dir -1 $genomeFolder | xargs basename -s .faa); do
    echo "Finding transporters for $g"
    gapseq find-transport -K $n_cores -b $gapseq_bits -f $modelsFolder/$g $genomeFolder/$g.faa \
    >> $modelsFolder/$g/log.log 2>> $modelsFolder/$g/log.err
done

Finding transporters for C_acetobutylicum
Finding transporters for C_beijerinckii_13821
Finding transporters for C_beijerinckii_791


Removing gene family annotation from the Transporter table to make the gene IDs match between the Transporter table and the filtered Reaction table.

In [8]:
dir -1 $genomeFolder | xargs basename -s .faa | xargs -I % -P $n_cores python -c \
"""
from pandas import read_table
df = read_table('$modelsFolder/%/%-Transporter.tbl', comment = '#')
df['stitle'] = df['stitle'].apply(lambda x: x.split(' ')[0])
df.to_csv('$modelsFolder/%/%-Transporter_fix.tbl', index = False, sep = '\t')
"""

In [9]:
for g in $(dir -1 $genomeFolder | xargs basename -s .faa); do
    mv $modelsFolder/$g/$g-Transporter_fix.tbl tmp
    cat <(grep '#' $modelsFolder/$g/$g-Transporter.tbl) tmp > $modelsFolder/$g/$g-Transporter_fix.tbl
    rm tmp
done

In [10]:
dir -1 $genomeFolder | xargs basename -s .faa | xargs -I % -P $n_cores bash -c \
"""
echo 'Drafting a model for %'
$gapseq_exc draft -r $modelsFolder/%/filtered-Reactions.tbl \
-t $modelsFolder/%/%-Transporter_fix.tbl \
-b $biomass -u $gapseq_bits -f $modelsFolder/% \
>> $modelsFolder/%/log.log 2>> $modelsFolder/%/log.err
"""

Drafting a model for C_acetobutylicum
Drafting a model for C_beijerinckii_13821
Drafting a model for C_beijerinckii_791


### Gapfill draft strain model

In [11]:
dir -1 $genomeFolder | xargs basename -s .faa | xargs -I % -P $n_cores bash -c \
"""
echo 'Gapfilling the model of %'
$gapseq_exc fill -q -m $modelsFolder/%/filtered-Reactions.tbl-draft.RDS -n $medium \
-c $modelsFolder/%/filtered-Reactions.tbl-rxnWeights.RDS -g $modelsFolder/%/filtered-Reactions.tbl-rxnXgenes.RDS \
-f $modelsFolder/% >>$modelsFolder/%/log.log 2>>$modelsFolder/%/log.err
"""

Gapfilling the model of C_acetobutylicum
Gapfilling the model of C_beijerinckii_13821
Gapfilling the model of C_beijerinckii_791


In [12]:
conda deactivate